# Brain extraction(Skull stripping)

How to do brain extraction using registration.

In [1]:
import os
from helpers import *

import ants
import SimpleITK as sitk

print(f'AntsPy version = {ants.__version__}')
print(f'SimpleITK version = {sitk.__version__}')

AntsPy version = 0.6.3
SimpleITK version = 2.4.0


In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
print(f'project folder = {BASE_DIR}')

project folder = /home/folkk2/img_group_project/test-MRI-preprocessing-technique


In [3]:
raw_examples = [
    'fsl-open-dev_sub-001_T1w.nii.gz',
    'wash-120_sub-001_T1w.nii.gz',
    'kf-panda_sub-01_ses-3T_T1w.nii.gz',
    'listen-task_sub-UTS01_ses-1_T1w.nii.gz',
]

### Raw Image

In [4]:
raw_example = raw_examples[0]
raw_img_path = os.path.join(BASE_DIR, 'assets', 'raw_examples', raw_example)
raw_img_ants = ants.image_read(raw_img_path, reorient='IAL')

explore_3D_array(arr=raw_img_ants.numpy(), cmap='nipy_spectral')

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

### Template based method (Native space)

#### Template Image

In [5]:
template_img_path = os.path.join(BASE_DIR, 'assets', 'templates', 'mni_icbm152_t1_tal_nlin_sym_09a.nii')
template_img_ants = ants.image_read(template_img_path, reorient='IAL')

explore_3D_array(arr = template_img_ants.numpy())

interactive(children=(IntSlider(value=94, description='SLICE', max=188), Output()), _dom_classes=('widget-inte…

### Brain Mask of the template

In [6]:
mask_template_img_path = os.path.join(BASE_DIR, 'assets', 'templates', 'mni_icbm152_t1_tal_nlin_sym_09a_mask.nii')
mask_template_img_ants = ants.image_read(mask_template_img_path, reorient='IAL')

explore_3D_array(mask_template_img_ants.numpy())

interactive(children=(IntSlider(value=94, description='SLICE', max=188), Output()), _dom_classes=('widget-inte…

In [7]:
np.unique(mask_template_img_ants.numpy())

array([0., 1.], dtype=float32)

### Register template to raw image

In [8]:
transformation = ants.registration(
    fixed=raw_img_ants,
    moving=template_img_ants, 
    type_of_transform='SyN',
    verbose=True
)

antsRegistration --dimensionality 3 -r [0x77da20b6da48,0x77da20b6df08,1] --metric mattes[0x77da20b6da48,0x77da20b6df08,1,32,regular,0.2] --transform Affine[0.25] --convergence 2100x1200x1200x0 --smoothing-sigmas 3x2x1x0 --shrink-factors 4x2x2x1 -x [NA,NA] --metric mattes[0x77da20b6da48,0x77da20b6df08,1,32] --transform SyN[0.200000,3.000000,0.000000] --convergence [40x20x0,1e-7,8] --smoothing-sigmas 2x1x0 --shrink-factors 4x2x1 -u 0 -z 1 --output [/tmp/tmp_0txpdmc,0x77da20b6c6c8,0x77da20b6c6a8] -x [NA,NA] --float 1 --write-composite-transform 0 -v 1
All_Command_lines_OK
Using single precision for computations.
The composite transform comprises the following transforms (in order): 
  1. Center of mass alignment using fixed image: 0x77da20b6da48 and moving image: 0x77da20b6df08 (type = Euler3DTransform)
  Reading mask(s).
    Registration stage 0
      No fixed mask
      No moving mask
    Registration stage 1
      No fixed mask
      No moving mask
  number of levels = 4
  number of le

In [9]:
print(transformation)

{'warpedmovout': ANTsImage (IAL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (192, 192, 160)
	 Spacing    : (1.25, 1.25, 1.2)
	 Origin     : (98.1114, -149.1525, -129.5975)
	 Direction  : [ 0.  0. -1.  0.  1.  0.  1.  0.  0.]
, 'warpedfixout': ANTsImage (IAL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (189, 233, 197)
	 Spacing    : (1.0, 1.0, 1.0)
	 Origin     : (98.0, -98.0, -72.0)
	 Direction  : [ 0.  0. -1.  0.  1.  0.  1.  0.  0.]
, 'fwdtransforms': ['/tmp/tmp_0txpdmc1Warp.nii.gz', '/tmp/tmp_0txpdmc0GenericAffine.mat'], 'invtransforms': ['/tmp/tmp_0txpdmc0GenericAffine.mat', '/tmp/tmp_0txpdmc1InverseWarp.nii.gz']}


In [10]:
registered_img_ants = transformation['warpedmovout']

explore_3D_array_comparison(
    arr_before=raw_img_ants.numpy(), 
    arr_after=registered_img_ants.numpy()
)

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

### Apply the generated transformations to the mask of template

In [11]:
brain_mask = ants.apply_transforms(
    fixed=transformation['warpedmovout'],
    moving=mask_template_img_ants,
    transformlist=transformation['fwdtransforms'],
    interpolator='nearestNeighbor',
    verbose=True
)

['-d', '3', '-i', '0x77da2076a2c8', '-o', '0x77da20b6dea8', '-r', '0x77da2076b068', '-n', 'nearestNeighbor', '-t', '/tmp/tmp_0txpdmc1Warp.nii.gz', '-t', '/tmp/tmp_0txpdmc0GenericAffine.mat']
Using double precision for computations.
Default pixel value: 0
Input scalar image: 0x77da2076a2c8
Input is not a file, assuming dimension = 3 and scalar pixel type
Reference image: 0x77da2076b068
The composite transform comprises the following transforms (in order): 
  1. /tmp/tmp_0txpdmc0GenericAffine.mat (type = AffineTransform)
  2. /tmp/tmp_0txpdmc1Warp.nii.gz (type = DisplacementFieldTransform)
Interpolation type: NearestNeighborInterpolateImageFunction
Output warped image: 0x77da20b6dea8


In [12]:
explore_3D_array(brain_mask.numpy())

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

In [13]:
explore_3D_array_with_mask_contour(raw_img_ants.numpy(), brain_mask.numpy())

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

In [14]:
brain_mask_dilated = ants.morphology(brain_mask, radius=4, operation='dilate', mtype='binary')

explore_3D_array_with_mask_contour(raw_img_ants.numpy(), brain_mask_dilated.numpy())

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

### Save brain mask

In [16]:
out_folder =  os.path.join(BASE_DIR, 'assets', 'preprocessed')
out_folder = os.path.join(out_folder, raw_example.split('.')[0]) # create folder with name of the raw file
os.makedirs(out_folder, exist_ok=True) # create folder if not exists

out_filename = add_suffix_to_filename(raw_example, suffix='brainMaskByTemplate')
out_path = os.path.join(out_folder, out_filename)

print(raw_img_path[len(BASE_DIR):])
print(out_path[len(BASE_DIR):])

/assets/raw_examples/fsl-open-dev_sub-001_T1w.nii.gz
/assets/preprocessed/fsl-open-dev_sub-001_T1w/fsl-open-dev_sub-001_T1w_brainMaskByTemplate.nii.gz


In [17]:
brain_mask_dilated.to_file(out_path)

### Generate brain masked

In [18]:
masked = ants.mask_image(raw_img_ants, brain_mask_dilated)

explore_3D_array(masked.numpy())

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

In [19]:
out_filename = add_suffix_to_filename(raw_example, suffix='brainMaskedByTemplate')
out_path = os.path.join(out_folder, out_filename)

print(raw_img_path[len(BASE_DIR):])
print(out_path[len(BASE_DIR):])

/assets/raw_examples/fsl-open-dev_sub-001_T1w.nii.gz
/assets/preprocessed/fsl-open-dev_sub-001_T1w/fsl-open-dev_sub-001_T1w_brainMaskedByTemplate.nii.gz


In [20]:
masked.to_file(out_path)